# **LLC, grokking and weight matching: practice**

Notebook for the module "Developmental interpretability". It collects the toy runs of the module's lessons, and the numbers in the lesson steps come from its output:

* lesson "Singular Learning Theory and the LLC": the volume of near-optimal weights $V(\varepsilon)$ and its slope on log-log axes for three loss functions, and an LLC estimate with Langevin dynamics on the same functions;
* lesson "Phase transitions in training": grokking of addition $(a+b)\bmod 17$ and the Fourier concentration of the embedding $C(t)$ over training steps;
* lesson "Weight-space interpretability": the interpolation barrier before and after weight matching, for a permuted copy of a network and for pairs of independently trained networks; the partition of the plane by a two-dimensional ReLU network into polytopes.

A CPU is enough: all models are small; on three CPU threads our run of the whole notebook took 3 minutes. Every cell sets its seeds, so the numbers repeat on a rerun. The exercises are numbered as in the steps of the lesson "LLC, grokking and weight matching: Practice".

## **Setup**
Only torch, numpy, scipy and matplotlib are needed, and Colab already has them. Everything runs on the CPU, so the numbers match the lesson steps on a GPU machine as well.

In [ ]:
import math, json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

torch.set_default_dtype(torch.float64)
R = {}  # all numbers of the notebook; saved to toys_33_results.json at the end
T0 = time.time()
print("torch", torch.__version__)

## **1. Volume of near-optimal weights and its slope**

Three loss functions on $\mathbb R^2$ with minimum $L = 0$ at the origin: $w_1^2 + w_2^2$ (theory $\lambda = 1$), $w_1^2 + w_2^4$ ($\lambda = 0.75$) and $w_1^2 w_2^4$ ($\lambda = 1/4$). The volume $V(\varepsilon)$ of the region $L \le \varepsilon$ inside the square $[-1,1]^2$ is the fraction of $2^{25} = 33\,554\,432$ uniform points times the area of the square, 4. The slope is a least-squares fit over five points of $\log V$ against $\log \varepsilon$. The same procedure in the cube $[-1,1]^3$ (volume 8) gives the slope for $w_1^2 + w_2^4 + w_3^6$ from the lesson exercise, where theory gives $\lambda = 11/12$.

In [ ]:
N_POINTS = 2**25
EPS = [1e-2, 3e-3, 1e-3, 3e-4, 1e-4]
LOSSES = {
    "w1^2 + w2^2": (2, lambda w: w[:, 0]**2 + w[:, 1]**2),
    "w1^2 + w2^4": (2, lambda w: w[:, 0]**2 + w[:, 1]**4),
    "w1^2 * w2^4": (2, lambda w: w[:, 0]**2 * w[:, 1]**4),
    "w1^2 + w2^4 + w3^6": (3, lambda w: w[:, 0]**2 + w[:, 1]**4 + w[:, 2]**6),
}

def volumes(loss, d, seed=0):
    g = torch.Generator().manual_seed(seed)
    w = (torch.rand(N_POINTS, d, generator=g, dtype=torch.float32) * 2 - 1).double()
    L = loss(w)
    return [2**d * (L <= e).double().mean().item() for e in EPS]

def slope(V):
    return float(np.polyfit(np.log(EPS), np.log(V), 1)[0])

R["volume"] = {}
for name, (d, loss) in LOSSES.items():
    V = volumes(loss, d)
    R["volume"][name] = {"V": V, "slope": slope(V)}

print("eps     " + "".join(f"{name:>22}" for name in LOSSES))
for i, e in enumerate(EPS):
    print(f"{e:<8.0e}" + "".join(f"{R['volume'][n]['V'][i]:>22.4g}" for n in LOSSES))
print("slope   " + "".join(f"{R['volume'][n]['slope']:>22.3f}" for n in LOSSES))
print("pi*eps at 1e-2:", round(math.pi * 1e-2, 5), "  exact volume of w1^2*w2^4 at 1e-2:", round(8 * 1e-2**0.25 - 4 * 1e-2**0.5, 4))

**Exercise 1. What slope of $\log V$ against $\log\varepsilon$ did the run give for $L = w_1^2 w_2^4$? Give the answer with three decimals.**

The slope is below the theoretical $1/4$: the exact volume of this region in the square is $8\varepsilon^{1/4} - 4\varepsilon^{1/2}$, and the second term still matters on this range of $\varepsilon$.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name in ["w1^2 + w2^2", "w1^2 + w2^4", "w1^2 * w2^4"]:
    ax.loglog(EPS, R["volume"][name]["V"], "o-", label=f"{name}: slope {R['volume'][name]['slope']:.3f}")
ax.set_xlabel("epsilon"); ax.set_ylabel("V(epsilon)"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## **2. LLC estimate with Langevin dynamics**

The estimator of Lau et al. (2023): $\hat\lambda(w^\star) = n\beta^\ast\big(\mathbb E[L_n(w)] - L_n(w^\star)\big)$, $\beta^\ast = 1/\log n$, with the mean over the distribution $\propto \exp\{-n\beta^\ast L_n(w) - \tfrac{\gamma}{2}\|w - w^\star\|^2\}$. Here $L_n = L$, $w^\star = 0$, $n = 10^4$, $\gamma = 1$. The samples come from Langevin dynamics with the full gradient:

$$w \leftarrow w - \tfrac{\epsilon}{2}\big(n\beta^\ast \nabla L(w) + \gamma (w - w^\star)\big) + \mathcal N(0, \epsilon I),\qquad \epsilon = 10^{-5},$$

1000 chains of 40,000 steps, the first 10,000 discarded, seed 0. About a minute on a CPU.

*The paper and the library differ here. In Lau et al., $n\beta^\ast = n/\log n$, where $n$ is the sample size, while `devinterp` recommends `default_nbeta(dataloader)`, the batch size divided by its logarithm; without an explicit value, the optimizer `SGLD` takes `nbeta=1` and warns that the estimate will be wrong. Here it is as in the paper.*

In [ ]:
def llc(loss, d, n=10**4, gamma=1.0, eps=1e-5, chains=1000, steps=40000, burn=10000, seed=0):
    torch.manual_seed(seed)
    nbeta = n / math.log(n)
    w = torch.zeros(chains, d, requires_grad=True)
    total, count = 0.0, 0
    for t in range(steps):
        L = loss(w)
        grad, = torch.autograd.grad(L.sum(), w)
        with torch.no_grad():
            w += -eps / 2 * (nbeta * grad + gamma * w) + math.sqrt(eps) * torch.randn_like(w)
        if t >= burn:
            total += L.detach().mean().item(); count += 1
    return nbeta * total / count  # L(w*) = 0

R["llc"] = {}
for name, (d, loss) in LOSSES.items():
    R["llc"][name] = llc(loss, d)
    print(f"{name:<20} LLC estimate {R['llc'][name]:.3f}   volume slope {R['volume'][name]['slope']:.3f}")

**Exercise 2. What estimate $\hat\lambda$ did the run give for $L = w_1^2 + w_2^4$? Give the answer with three decimals.**

For a sum of powers of different coordinates, the mean of $n\beta^\ast L$ under this distribution equals $\lambda$ for any $n$, so the first two estimates differ from theory only by sampling noise. For $w_1^2 w_2^4$ the loss is zero on the axes, the distribution drifts along them until the localization term $\gamma$ stops it, and the estimate is biased downward.

## **3. Grokking addition modulo 17**

The task is $(a+b)\bmod 17$; the input is the concatenated one-hot vectors of $a$ and $b$; the model is a two-layer perceptron of width 48 with a quadratic activation $f(x) = \big((xW_1)^2\big)W_2$, weights $W_1$, $W_2$ drawn from $\mathcal N(0,1)$ and divided by the square root of the layer's number of inputs. The error is MSE to the one-hot answer, averaged over all outputs; full batch, AdamW with learning rate $3\cdot10^{-3}$ and weight decay 1, 60,000 steps. Training uses the first 144 pairs of the seed's `randperm` permutation, the test uses the other 145 of 289; float64, seeds 0–4 (the seed sets both the split and the initialization).

The structural measure is the Fourier concentration of the embedding of $a$, that is, of the rows of $W_1$ for one-hot $a$ (a $17\times 48$ matrix). Along the cyclic axis of $a$ we take the discrete Fourier transform and the power $P_{nk}$ of neuron $n$ at frequency $k = 1,\dots,8$ (frequencies $k$ and $17-k$ are added, the constant term is dropped):

$$C(t) = \frac{\sum_n \max_k P_{nk}(t)}{\sum_n\sum_k P_{nk}(t)}.$$

One run takes about 15 seconds; ten runs (five seeds at two learning rates) take two to three minutes.

In [ ]:
P_MOD, WIDTH, STEPS = 17, 48, 60000
A_ = torch.arange(P_MOD).repeat_interleave(P_MOD); B_ = torch.arange(P_MOD).repeat(P_MOD)
X_ADD = torch.cat([torch.nn.functional.one_hot(A_, P_MOD), torch.nn.functional.one_hot(B_, P_MOD)], 1).double()
Y_ADD = (A_ + B_) % P_MOD
T_ADD = torch.nn.functional.one_hot(Y_ADD, P_MOD).double()

def fourier_concentration(W1):
    power = torch.fft.fft(W1[:P_MOD], dim=0).abs()**2   # 17 x 48
    power = power[1:9] + power[9:17].flip(0)               # frequencies k and 17-k together
    return (power.max(0).values.sum() / power.sum()).item()

def grok(seed, lr=3e-3, steps=STEPS, log_every=100):
    torch.manual_seed(seed)
    perm = torch.randperm(P_MOD * P_MOD); tr, te = perm[:144], perm[144:]
    W1 = (torch.randn(2 * P_MOD, WIDTH) / (2 * P_MOD)**0.5).requires_grad_()
    W2 = (torch.randn(WIDTH, P_MOD) / WIDTH**0.5).requires_grad_()
    opt = torch.optim.AdamW([W1, W2], lr=lr, weight_decay=1.0)
    hist, first_train, first_test = [], None, None
    for t in range(steps + 1):
        out = ((X_ADD @ W1)**2) @ W2
        loss = ((out[tr] - T_ADD[tr])**2).mean()
        with torch.no_grad():
            pred = out.argmax(1)
            acc_tr = (pred[tr] == Y_ADD[tr]).double().mean().item()
            acc_te = (pred[te] == Y_ADD[te]).double().mean().item()
            if first_train is None and acc_tr == 1: first_train = t
            if first_test is None and acc_te == 1: first_test = t
            if t % log_every == 0:
                hist.append([t, acc_tr, acc_te, loss.item(), fourier_concentration(W1)])
        if t == steps: break
        opt.zero_grad(); loss.backward(); opt.step()
    return {"hist": hist, "first_train": first_train, "first_test": first_test}

R["grok"] = {str(s): grok(s) for s in range(5)}
print("seed 0:")
print(f"{'step':>6} {'acc train':>10} {'acc test':>9} {'MSE train':>10} {'C(t)':>6}")
for t, a_tr, a_te, mse, c in R["grok"]["0"]["hist"]:
    if t in (0, 500, 5000, 10000, 20000, 60000):
        print(f"{t:>6} {a_tr:>10.3f} {a_te:>9.3f} {mse:>10.1e} {c:>6.3f}")
print()
for s in range(5):
    g = R["grok"][str(s)]
    print(f"seed {s}: train accuracy 1.000 from step {g['first_train']}, test accuracy 1.000 from step {g['first_test']}, C at step 60000 = {g['hist'][-1][4]:.3f}")

**Exercise 3. From which step does seed 0 reach test accuracy 1.000?**

**Exercise 4. Which interval contains $C$ at step 60,000 across the five seeds?**

In [ ]:
h = np.array(R["grok"]["0"]["hist"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h[:, 0], h[:, 1], label="train accuracy"); ax.plot(h[:, 0], h[:, 2], label="test accuracy"); ax.plot(h[:, 0], h[:, 4], label="Fourier concentration C(t)")
ax.set_xscale("symlog", linthresh=100); ax.set_xlabel("training step"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

The same setup with learning rate $10^{-3}$: the length of each grokking phase depends on the learning rate and weight decay.

In [ ]:
R["grok_lr1e-3"] = {}
for s in range(5):
    g = grok(s, lr=1e-3, log_every=STEPS)
    R["grok_lr1e-3"][str(s)] = {"first_train": g["first_train"], "first_test": g["first_test"]}
    print(f"lr 1e-3, seed {s}: train accuracy 1.000 from step {g['first_train']}, test accuracy 1.000 from step {g['first_test']}")
print("seeds with test accuracy 1.000 within 60000 steps:", sum(v["first_test"] is not None for v in R["grok_lr1e-3"].values()), "of 5")

## **4. Interpolation barrier before and after weight matching**

The network is $f(x) = W_2\,\mathrm{ReLU}(W_1 x + b_1) + b_2$. The path between models is $\theta(\alpha) = \alpha\,\theta_A + (1-\alpha)\,\theta_B$, so $\alpha = 0$ is model $B$ and $\alpha = 1$ is model $A$. The barrier of Entezari et al. is the largest excess, over the points $\alpha$, of the loss over the linear interpolation of the endpoint losses. Weight matching for one hidden layer looks for the permutation of $B$'s neurons that maximizes the sum of inner products of the weights of matched neurons ($W_1$, $b_1$ and the columns of $W_2$): this is an assignment problem, solved by `linear_sum_assignment`.

**Experiment 1: a permuted copy.** Input dimension 4, $h = 6$ hidden neurons, output dimension 2, all weights of $A$ from $\mathcal N(0,1)$, seed 0. Neuron $i$ of model $B$ is neuron $P_{\text{true}}(i)$ of model $A$, $P_{\text{true}} = (2,4,1,3,5,0)$. The target is the output of $A$ on 1000 Gaussian inputs, the loss is MSE, so $L(\theta_A) = L(\theta_B) = 0$.

In [ ]:
def net(th, x):
    W1, b1, W2, b2 = th
    return torch.relu(x @ W1.T + b1) @ W2.T + b2

def path(a, b, alpha):  # alpha = 1 -> model a, alpha = 0 -> model b
    return [alpha * u + (1 - alpha) * v for u, v in zip(a, b)]

def permute(th, P):
    W1, b1, W2, b2 = th
    return [W1[P], b1[P], W2[:, P], b2]

def weight_matching(a, b):
    cost = a[0] @ b[0].T + torch.outer(a[1], b[1]) + a[2].T @ b[2]
    rows, cols = linear_sum_assignment(cost.numpy(), maximize=True)
    return torch.as_tensor(cols)

torch.manual_seed(0)
model_A = [torch.randn(6, 4), torch.randn(6), torch.randn(2, 6), torch.randn(2)]
P_TRUE = torch.tensor([2, 4, 1, 3, 5, 0])
model_B = permute(model_A, P_TRUE)
X1 = torch.randn(1000, 4); Y1 = net(model_A, X1)
mse1 = lambda th: ((net(th, X1) - Y1)**2).mean().item()

ALPHAS1 = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
R["exp1"] = {"max_abs_diff_fA_fB": (net(model_A, X1) - net(model_B, X1)).abs().max().item(),
             "alphas": ALPHAS1, "mse_before": [mse1(path(model_A, model_B, a)) for a in ALPHAS1]}
pi = weight_matching(model_A, model_B)
model_B_aligned = permute(model_B, pi)
R["exp1"]["matching"] = pi.tolist()
R["exp1"]["max_abs_W1_diff_after"] = (model_A[0] - model_B_aligned[0]).abs().max().item()
R["exp1"]["max_mse_after"] = max(mse1(path(model_A, model_B_aligned, a)) for a in np.linspace(0, 1, 11))
print("max |f_A - f_B| on the inputs:", f"{R['exp1']['max_abs_diff_fA_fB']:.1e}")
print("alpha:            ", "  ".join(f"{a:>6.1f}" for a in ALPHAS1))
print("MSE, no alignment:", "  ".join(f"{m:>6.3f}" for m in R["exp1"]["mse_before"]))
print("weight matching permutation:", R["exp1"]["matching"])
print("max |W1_A - W1_B'| after matching:", R["exp1"]["max_abs_W1_diff_after"], "  max MSE on the path after matching:", f"{R['exp1']['max_mse_after']:.1e}")

**Experiment 2: two independently trained networks.** A teacher network $8\to32\to1$ (weights and biases from $\mathcal N(0,1)$, divided by the square root of the layer's number of inputs, seed 1) defines the target; students $8\to h\to 1$ with the default `nn.Linear` initialization are trained from different seeds: Adam, learning rate $10^{-3}$, 3000 steps, batch 256, 4096 training inputs. The loss is MSE on 4096 test inputs, the barrier uses 11 points $\alpha$, five pairs of models (seeds 100 and 101, 102 and 103, …, 108 and 109). The three values of $h$ take about a minute on a CPU.

In [ ]:
torch.manual_seed(1)
teacher = [torch.randn(32, 8) / 8**0.5, torch.randn(32) / 8**0.5, torch.randn(1, 32) / 32**0.5, torch.randn(1) / 32**0.5]
g = torch.Generator().manual_seed(2)
X_train = torch.randn(4096, 8, generator=g); X_test = torch.randn(4096, 8, generator=g)
Y_train, Y_test = net(teacher, X_train), net(teacher, X_test)
ALPHAS2 = np.linspace(0, 1, 11)

def student(h, seed):
    torch.manual_seed(seed)
    l1, l2 = torch.nn.Linear(8, h), torch.nn.Linear(h, 1)
    th = [p.data.clone().requires_grad_() for p in (l1.weight, l1.bias, l2.weight, l2.bias)]
    opt = torch.optim.Adam(th, lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(3000):
        idx = torch.randint(0, 4096, (256,), generator=g)
        loss = ((net(th, X_train[idx]) - Y_train[idx])**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return [p.detach() for p in th]

test_mse = lambda th: ((net(th, X_test) - Y_test)**2).mean().item()

def excess(a, b):  # loss on the path minus the linear interpolation of the endpoint losses
    la, lb = test_mse(a), test_mse(b)
    return [test_mse(path(a, b, al)) - (al * la + (1 - al) * lb) for al in ALPHAS2]

R["exp2"] = {}
print(f"{'h':>5} {'MSE A':>7} {'MSE B':>7}   barrier, no alignment     barrier after matching    ratio")
for h in [16, 128, 1024]:
    pairs = []
    for k in range(5):
        a, b = student(h, 100 + 2 * k), student(h, 101 + 2 * k)
        b_aligned = permute(b, weight_matching(a, b))
        pairs.append({"mse_A": test_mse(a), "mse_B": test_mse(b), "excess_before": excess(a, b), "excess_after": excess(a, b_aligned)})
    before = np.array([max(p["excess_before"]) for p in pairs]); after = np.array([max(p["excess_after"]) for p in pairs])
    R["exp2"][str(h)] = {"pairs": pairs, "barrier_before": before.tolist(), "barrier_after": after.tolist()}
    print(f"{h:>5} {np.mean([p['mse_A'] for p in pairs]):>7.4f} {np.mean([p['mse_B'] for p in pairs]):>7.4f}"
          f"   {before.mean():.4f} [{before.min():.4f}, {before.max():.4f}]   {after.mean():.4f} [{after.min():.4f}, {after.max():.4f}]   {before.mean() / after.mean():.1f}")

**Exercise 5. By what factor does weight matching reduce the mean barrier at $h = 16$? Give the answer with one decimal.**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for h, c in [("16", "C0"), ("128", "C1"), ("1024", "C2")]:
    pairs = R["exp2"][h]["pairs"]
    ax.plot(ALPHAS2, np.mean([p["excess_before"] for p in pairs], 0), "--", color=c, label=f"h={h}, no alignment")
    ax.plot(ALPHAS2, np.mean([p["excess_after"] for p in pairs], 0), "-", color=c, label=f"h={h}, weight matching")
ax.set_xlabel("alpha (0 = model B, 1 = model A)"); ax.set_ylabel("excess test MSE"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()

## **5. Polytopes of a two-dimensional ReLU network**

A $2\to8\to8$ ReLU network, weights and biases from $\mathcal N(0,1)$, the second-layer weights divided by $\sqrt 8$, seed 0. The spline code of an input is 16 bits: which neurons of both layers are above threshold. On a grid of $801\times801$ points in the square $[-3,3]^2$ we count how many distinct codes occur: this is the number of polytopes that contain at least one grid point. The first-layer neurons cut the plane with lines $w_i^\top x + b_i = 0$; the second-layer neurons split these regions further along piecewise-linear boundaries. Permuting neurons permutes the bits of the code and changes no polytope.

In [ ]:
torch.manual_seed(0)
poly = {"W1": torch.randn(8, 2), "b1": torch.randn(8), "W2": torch.randn(8, 8) / 8**0.5, "b2": torch.randn(8)}
xs = torch.linspace(-3, 3, 801)
grid = torch.stack(torch.meshgrid(xs, xs, indexing="xy"), -1).reshape(-1, 2)

def spline_code(th, x):
    z1 = x @ th["W1"].T + th["b1"]
    z2 = torch.relu(z1) @ th["W2"].T + th["b2"]
    return torch.cat([z1 > 0, z2 > 0], 1)

def n_regions(bits):
    return len(torch.unique(bits.long() @ (2 ** torch.arange(bits.shape[1])), sorted=False))

codes = spline_code(poly, grid)
perm = torch.tensor([3, 0, 6, 1, 7, 2, 5, 4])
poly_perm = {"W1": poly["W1"][perm], "b1": poly["b1"][perm], "W2": poly["W2"][:, perm], "b2": poly["b2"]}
R["poly"] = {"weights": {k: v.tolist() for k, v in poly.items()},
             "regions_layer1": n_regions(codes[:, :8]), "regions_both": n_regions(codes), "regions_both_permuted": n_regions(spline_code(poly_perm, grid))}
print("regions cut by the 8 first-layer lines:", R["poly"]["regions_layer1"])
print("regions of the full 16-bit code:", R["poly"]["regions_both"], "  after permuting first-layer neurons:", R["poly"]["regions_both_permuted"])

**Exercise 6. How many polytopes of the full 16-bit code fall on the grid?**

In [ ]:
key = (codes.long() @ (2 ** torch.arange(16))).reshape(801, 801)
_, ids = torch.unique(key, return_inverse=True)
rng = np.random.default_rng(0)
colors = rng.random((ids.max().item() + 1, 3)) * 0.6 + 0.4
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(colors[ids.numpy()], extent=[-3, 3, -3, 3], origin="lower")
t = np.linspace(-3, 3, 2)
for (w_x, w_y), b in zip(poly["W1"].tolist(), poly["b1"].tolist()):
    ax.plot(t, -(w_x * t + b) / w_y, "k-", lw=0.8)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_title("polytopes of the 2-8-8 ReLU network")
plt.show()

json.dump(R, open("toys_33_results.json", "w"))
print("saved toys_33_results.json; total time", round(time.time() - T0), "s")

## **Summary**

* The slope of the volume of near-optimal weights drops with degeneracy: 1, 0.75 and 1/4 in theory at the same $d = 2$; the LLC estimator reproduces this on the regular and the quasi-homogeneous loss and is biased downward where the loss is zero on whole axes.
* In grokking, training accuracy reaches 1 within about a hundred steps, while test accuracy and Fourier concentration rise together over thousands of steps after that.
* A high barrier between two networks can come from a single permutation of neurons; weight matching removes it entirely for the permuted copy and reduces it several times for independently trained networks.